# Notebook 2 — Oracle Retrain on Retain Set

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv:2604.08271v1)

**Purpose:** Train a fresh model from scratch on the **retain set only** per (seed, ratio),
using the SAME paper-faithful hyperparameters as Θ_o (Notebook 1).  
The oracle retrain is the gold-standard baseline for unlearning.

**Prerequisites:** Run Notebook 1 first. Set `CKPT_DATASET_DIR` to the Kaggle dataset path.

**Outputs per (seed, ratio):**
- `oracle_ratio{30|10}_seed{seed}.pt` — self-describing checkpoint with all metrics
- `results_oracle.csv` — Output/Probe/NCC retain+forget accuracy

**Convention:** forget accuracy compared against oracle's own forget acc (NOT against 0%).  
Oracle reaches non-zero forget acc due to feature transferability — documented in §4.1.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU    :', torch.cuda.get_device_name(0))

In [ ]:
# A.8 fix: clone official repo + record commit
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/ycgao1/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
# ══ SET THIS to the Kaggle dataset mount path from Notebook 1 ══
CKPT_DATASET_DIR = '/kaggle/input/datasets/kiethe/cmf-notebook1'  # ← EDIT if your slug differs

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
]

config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path  = _p
        CKPT_ROOT_NB1 = os.path.dirname(_p)
        break
assert config_path, f'Cannot find cmf_benchmark_config.json under {CKPT_DATASET_DIR}'

with open(config_path) as f:
    NB1_CFG = json.load(f)

print('Config loaded from:', config_path)
print('NB1 commit :', NB1_CFG.get('repo_commit', 'unknown'))

DATASET     = NB1_CFG['dataset']
ARCH        = NB1_CFG['arch']
NUM_CLASSES = NB1_CFG['num_classes']
SEEDS       = NB1_CFG['seeds']
RATIOS      = NB1_CFG['ratios']
TEST_MODE   = NB1_CFG.get('test_mode', False)

# Oracle retrain config: same optimizer as pretrain but DIFFERENT epochs.
# Paper Table 4 line 2314: Retain-only Retrain CIFAR-10 = 200 epochs (NOT 300).
PT = NB1_CFG['pretrain']
LR_INIT       = PT['lr_init']       # 1e-2 (Table 4: same for retrain)
WEIGHT_DECAY  = PT['weight_decay']
MOMENTUM      = PT['momentum']
WARMUP_EPOCHS = PT['warmup_epochs']
MIN_LR        = PT['min_lr']
BATCH_SIZE    = PT['batch_size']
# Table 4: Retain-only Retrain CIFAR-10/100 = 200 epochs, NOT 300
EPOCHS        = 5 if TEST_MODE else 200   # Table 4 line 2314
PATIENCE      = PT['patience']

CKPT_ROOT  = '/kaggle/working/checkpoints/oracle'
os.makedirs(CKPT_ROOT, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DATASET={DATASET}  ARCH={ARCH}  SEEDS={SEEDS}  RATIOS={RATIOS}  device={device}')

In [ ]:
import torchvision, torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                          download=True, transform=transform_train)
test_set   = torchvision.datasets.CIFAR10('/kaggle/working/data', train=False,
                                          download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256,
                                          shuffle=False, num_workers=2)
print(f'Train: {len(full_train)}  Test: {len(test_set)}')

In [ ]:
from models.resnet import ResNet18

def build_model():
    return ResNet18(num_classes=NUM_CLASSES, dataset=DATASET).to(device)

def make_scheduler(optimizer, warmup_epochs, total_epochs, min_lr):
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=min_lr)
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])

@torch.no_grad()
def accuracy_on_loader(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)

@torch.no_grad()
def extract_features(model, loader):
    """Extract avgpool features via forward hook — works for both
    ResNet_cifar (no maxpool) and standard ResNet (has maxpool)."""
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        buf = []
        hook = model.avgpool.register_forward_hook(
            lambda m, i, o: buf.append(o.flatten(1).detach().cpu())
        )
        model(x)
        hook.remove()
        feats.append(buf[0])
        labs.append(y)
    return torch.cat(feats), torch.cat(labs)

def run_linear_probe(model, train_loader, test_loader, n_epochs=50, lr=1e-2):
    Xtr, ytr = extract_features(model, train_loader)
    Xte, yte = extract_features(model, test_loader)
    d = Xtr.size(1)
    head = nn.Linear(d, NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=lr, momentum=0.9)
    ds   = torch.utils.data.TensorDataset(Xtr, ytr)
    ldr  = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad():
        pred = head(Xte.to(device)).argmax(1).cpu()
    return pred, yte

def ncc_classify(model, train_loader, test_loader):
    Xtr, ytr = extract_features(model, train_loader)
    Xte, yte = extract_features(model, test_loader)
    Xtr_n = F.normalize(Xtr, dim=1)
    means  = torch.zeros(NUM_CLASSES, Xtr_n.size(1))
    for c in range(NUM_CLASSES):
        m = ytr == c
        if m.any(): means[c] = Xtr_n[m].mean(0)
    means_n = F.normalize(means, dim=1)
    Xte_n   = F.normalize(Xte, dim=1)
    pred    = (Xte_n @ means_n.t()).argmax(1)
    return pred, yte

def split_acc(pred, true, forget_mask):
    retain_mask = ~forget_mask
    ret = (pred[retain_mask] == true[retain_mask]).float().mean().item() * 100
    fgt = (pred[forget_mask] == true[forget_mask]).float().mean().item() * 100
    return ret, fgt

print('Helpers ready.')

In [ ]:
# A.6 fix: Load the SAME split files generated by NB1 — never re-split here.

all_results = []

for ratio in RATIOS:
    for seed in SEEDS:
        tag = f'ratio{ratio}_seed{seed}'
        if TEST_MODE: tag += '_testmode'
        ckpt_path = f'{CKPT_ROOT}/oracle_{tag}.pt'

        # ── Load split (A.6: always from NB1) ─────────────────────────────
        fpath = f'{CKPT_ROOT_NB1}/splits/forget_indices_{tag}.json'
        rpath = f'{CKPT_ROOT_NB1}/splits/retain_indices_{tag}.json'
        assert os.path.exists(fpath), f'Split file not found: {fpath}'
        with open(fpath) as f: forget_idx = json.load(f)
        with open(rpath) as f: retain_idx = json.load(f)

        retain_set     = torch.utils.data.Subset(full_train, retain_idx)
        retain_loader  = torch.utils.data.DataLoader(retain_set,
                             batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        full_loader    = torch.utils.data.DataLoader(full_train,
                             batch_size=256, shuffle=False, num_workers=2)

        # ── Build forget mask for test set evaluation ──────────────────────
        # For random-mix split: a sample is 'forgotten' if its index is in forget_idx.
        # On the test set we use class-level matching (all test samples of classes
        # that were partly forgotten); here we do simpler: evaluate on full test set.
        test_targets = torch.tensor(test_set.targets)
        # forget classes = unique classes in forget set
        forget_classes_in_split = list(set(full_train.targets[i] for i in forget_idx))
        forget_test_mask = torch.tensor([t in set(forget_classes_in_split)
                                          for t in test_set.targets])

        # ── Check for existing checkpoint (resumable) ──────────────────────
        if os.path.exists(ckpt_path):
            print(f'[{tag}] Checkpoint exists — loading.')
            ck = torch.load(ckpt_path, map_location=device)
            model = build_model()
            model.load_state_dict(ck['model_state_dict'])
            wall_clock_minutes = 0.0   # A.2 fix: 0.0 when checkpoint-skip
            all_results.append(ck['metrics'])
            print(f'  output_retain={ck["metrics"]["output_retain_acc"]:.2f}%  '
                  f'output_forget={ck["metrics"]["output_forget_acc"]:.2f}%')
            continue

        # ── Train oracle from scratch ──────────────────────────────────────
        print(f'\n[{tag}] Training oracle retrain...')
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        model = build_model()
        optimizer = optim.SGD(model.parameters(), lr=LR_INIT,
                              momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True)
        scheduler = make_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS, MIN_LR)

        val_subset  = torch.utils.data.Subset(retain_set,
                          range(max(1, len(retain_set) - 5000), len(retain_set)))
        val_loader  = torch.utils.data.DataLoader(val_subset, batch_size=256,
                          shuffle=False, num_workers=2)

        best_val  = 0.0
        best_state = None
        patience_cnt = 0
        early_stopped_at = None
        log = []

        t_start = time.time()
        for epoch in range(1, EPOCHS + 1):
            model.train()
            for x, y in retain_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                F.cross_entropy(model(x), y).backward()
                optimizer.step()
            scheduler.step()

            val_acc = accuracy_on_loader(model, val_loader)
            log.append({'epoch': epoch, 'val_acc': val_acc})
            if val_acc > best_val:
                best_val   = val_acc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
            if epoch % 50 == 0:
                print(f'  epoch {epoch:3d}: val={val_acc:.2f}%  patience={patience_cnt}/{PATIENCE}')
            if patience_cnt >= PATIENCE:
                early_stopped_at = epoch
                print(f'  Early stop at {epoch}')
                break

        # A.2 fix: wall_clock_minutes correctly populated when actually trained
        wall_clock_minutes = (time.time() - t_start) / 60.0

        model.load_state_dict(best_state)

        # ── 3-metric evaluation ────────────────────────────────────────────
        out_all_acc = accuracy_on_loader(model, test_loader)

        # Output retain/forget from test split
        model.eval()
        with torch.no_grad():
            all_preds = []
            for x, _ in test_loader:
                all_preds.append(model(x.to(device)).argmax(1).cpu())
        preds = torch.cat(all_preds)
        true  = test_targets
        out_ret = (preds[~forget_test_mask] == true[~forget_test_mask]).float().mean().item() * 100
        out_fgt = (preds[forget_test_mask]  == true[forget_test_mask]).float().mean().item() * 100

        # Linear Probe
        lp_pred, lp_true = run_linear_probe(model, full_loader, test_loader)
        lp_ret = (lp_pred[~forget_test_mask] == lp_true[~forget_test_mask]).float().mean().item() * 100
        lp_fgt = (lp_pred[forget_test_mask]  == lp_true[forget_test_mask]).float().mean().item() * 100

        # NCC
        ncc_pred, ncc_true = ncc_classify(model, full_loader, test_loader)
        ncc_ret = (ncc_pred[~forget_test_mask] == ncc_true[~forget_test_mask]).float().mean().item() * 100
        ncc_fgt = (ncc_pred[forget_test_mask]  == ncc_true[forget_test_mask]).float().mean().item() * 100

        metrics = {
            'model': 'oracle', 'ratio': ratio, 'seed': seed, 'tag': tag,
            'output_retain_acc': out_ret, 'output_forget_acc': out_fgt,
            'probe_retain_acc': lp_ret,   'probe_forget_acc': lp_fgt,
            'ncc_retain_acc': ncc_ret,    'ncc_forget_acc': ncc_fgt,
            'wall_clock_minutes': wall_clock_minutes,
            'early_stopped_at': early_stopped_at,
        }

        # ── Save checkpoint (self-describing) ──────────────────────────────
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': {
                'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                'ratio': ratio, 'seed': seed,
                'lr_init': LR_INIT, 'weight_decay': WEIGHT_DECAY,
                'momentum': MOMENTUM, 'warmup_epochs': WARMUP_EPOCHS,
                'epochs': EPOCHS, 'patience': PATIENCE, 'min_lr': MIN_LR,
                'split_source': 'nb1',
                'repo_commit': REPO_COMMIT,
                'test_mode': TEST_MODE,
            },
            'seed': seed,
            'metrics': metrics,
            'train_log': log,
        }, ckpt_path)
        print(f'  Saved {ckpt_path}')
        print(f'  output R={out_ret:.2f}% F={out_fgt:.2f}%  '
              f'probe R={lp_ret:.2f}% F={lp_fgt:.2f}%  '
              f'ncc R={ncc_ret:.2f}% F={ncc_fgt:.2f}%')
        all_results.append(metrics)

# ── Summary ───────────────────────────────────────────────────────────────────
df = pd.DataFrame(all_results)
csv_path = f'{CKPT_ROOT}/results_oracle.csv'
df.to_csv(csv_path, index=False)
print('\n=== Oracle results (mean±std over seeds per ratio) ===')
for ratio in RATIOS:
    sub = df[df['ratio'] == ratio]
    print(f'\nRatio {ratio}:')
    for col in ['output_retain_acc','output_forget_acc','probe_retain_acc',
                'probe_forget_acc','ncc_retain_acc','ncc_forget_acc']:
        v = sub[col].dropna()
        if len(v): print(f'  {col:30s}: {v.mean():.2f} ± {v.std():.2f}')
print(f'\nResults saved: {csv_path}')